# 论文 19：咖啡自动机——深入理解不可逆性

**论文**：Scott Aaronson（2016），The Coffee Automaton

**核心问题**：为什么混合后的咖啡和牛奶无法自行重新分离？不可逆性揭示了计算、信息以及时间本身的哪些本质？

---

## 不可逆性之谜

将牛奶滴入咖啡，观察它扩散、旋转并最终混合均匀。现在尝试让这一过程反向发生——**你做不到。**

但这是一个谜题：
- 牛顿定律是**时间可逆的**：向后运行它们是完全有效的
- 分子间的每次碰撞都是可逆的
- 微观法则不偏向任何时间方向

那么**不可逆性从何而来？**

这不仅仅是物理学——它与以下因素密切相关：
- **计算**：我们可以逆向计算吗？
- **信息**：“忘记”的真正含义是什么？
- **机器学习**：为什么神经网络要压缩信息？
- **时间之箭**：为什么时间有方向？

---

## 本 Notebook 的实现内容

我们将从多个角度探讨不可逆性：

1. **咖啡混合**：扩散和熵增长
2. **相空间**：可逆性所在
3. **粗粒化**：我们如何丢失信息
4. **庞加莱回归**：咖啡理论上*终有一天*会重新分离
5. **麦克斯韦妖**：智力可以逆转熵吗？
6. **兰道尔原理**：信息擦除需要消耗能量
7. **计算不可逆性**：单向函数和哈希
8. **机器学习**：为什么神经网络会压缩并忘记
9. **第二定律**：统计力学视角
10. **现代认识**：计算的热力学

这远远超出了论文 1 对复杂性的介绍。我们将深入探讨**为什么**出现不可逆性以及它对计算和学习意味着什么。

让我们开始吧！

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.ndimage import convolve
from scipy.stats import entropy as scipy_entropy
from typing import Tuple, List, Callable
import hashlib
from dataclasses import dataclass

np.random.seed(42)

# 设置绘图样式
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries imported successfully!")
print("NumPy version:", np.__version__)
print("\nReady to explore irreversibility...")

# 第 1 部分：咖啡混合 - 扩散过程

让我们从经典的例子开始：牛奶扩散到咖啡中。

## 扩散方程

牛奶的浓度 $c(x, y, t)$ 根据以下公式演变：

$$\frac{\partial c}{\partial t} = D \nabla^2 c = D \left(\frac{\partial^2 c}{\partial x^2} + \frac{\partial^2 c}{\partial y^2}\right)$$

其中 $D$ 是**扩散系数**。

**核心认识**：这个方程具有明确的**时间箭头**。反向演化的行为完全不同：
- 向前：浓度分散（牛奶混合）
- 反向：浓度重新聚集（牛奶与咖啡重新分离）

但第二种情况违反了热力学第二定律！

In [ ]:
def initialize_coffee_cup(size: int = 64) -> np.ndarray:
    """初始化一个“咖啡杯”，中间滴一滴牛奶。
    
    返回：
        concentration：2D 数组，其中 1 = 牛奶，0 = 咖啡"""
    cup = np.zeros((size, size))
    
    # 在中间加一滴牛奶
    center = size // 2
    radius = size // 8
    
    y, x = np.ogrid[:size, :size]
    mask = (x - center)**2 + (y - center)**2 <= radius**2
    cup[mask] = 1.0
    
    return cup


def diffusion_step(concentration: np.ndarray, D: float = 0.1) -> np.ndarray:
    """使用有限差分的扩散的一个时间步长。
    
    参数：
        concentration：当前浓度场
        D：扩散系数
    
    返回：
        一个时间步后的新浓度"""
    # 拉普拉斯核（离散 ∇²）
    kernel = np.array([[0, 1, 0],
                       [1, -4, 1],
                       [0, 1, 0]])
    
    # 应用拉普拉斯算子
    laplacian = convolve(concentration, kernel, mode='constant', cval=0.0)
    
    # 更新：c(t+Δt) = c(t) + D·Δt·∇²c
    dt = 0.1
    new_concentration = concentration + D * dt * laplacian
    
    # 保持浓度在有效范围内
    new_concentration = np.clip(new_concentration, 0, 1)
    
    return new_concentration


def simulate_coffee_mixing(steps: int = 200, D: float = 0.1) -> List[np.ndarray]:
    """模拟咖啡随时间的混合。
    
    返回：
        每个时间步的浓度场列表"""
    cup = initialize_coffee_cup()
    history = [cup.copy()]
    
    for _ in range(steps):
        cup = diffusion_step(cup, D)
        history.append(cup.copy())
    
    return history


# 模拟混合
print("Simulating coffee mixing...\n")
mixing_history = simulate_coffee_mixing(steps=200)

# 显示关键帧
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
timesteps = [0, 25, 50, 100, 200]

for ax, t in zip(axes, timesteps):
    im = ax.imshow(mixing_history[t], cmap='RdYlBu_r', vmin=0, vmax=1)
    ax.set_title(f't = {t}')
    ax.axis('off')

plt.colorbar(im, ax=axes, label='Milk Concentration', fraction=0.046)
plt.suptitle('Coffee Mixing: Irreversible Diffusion', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("Observation: The milk spreads out and can never spontaneously unmix!")
print("\n✓ Diffusion simulation complete!")

# 第 2 节：熵增长 - 量化不可逆性

**熵**衡量无序性或“扩散性”。当牛奶混合时，熵会增加。

## 香农熵

将杯子划分成许多小区域，Shannon 熵为：

$$H = -\sum_i p_i \log_2 p_i$$

其中 $p_i$ 表示在第 $i$ 个区域找到牛奶的概率。

## 热力学熵

与宏观状态一致的微观状态$\Omega$的数量相关：

$$S = k_B \ln \Omega$$

**热力学第二定律**：在孤立的系统中，熵永远不会减少：

$$\frac{dS}{dt} \geq 0$$

这就是时间之箭！

In [ ]:
def compute_shannon_entropy(concentration: np.ndarray, num_bins: int = 10) -> float:
    """计算浓度分布的香农熵。
    
    参数：
        concentration：2D浓度场
        num_bins：直方图的箱数
    
    返回：
        香农熵（以位为单位）"""
    # 展平并创建直方图
    flat = concentration.flatten()
    hist, _ = np.histogram(flat, bins=num_bins, range=(0, 1), density=True)
    
    # 标准化为概率
    hist = hist / hist.sum()
    
    # 计算香农熵
    return scipy_entropy(hist, base=2)


def compute_spatial_entropy(concentration: np.ndarray) -> float:
    """计算空间熵（浓度方差）。
    
    随着混合的进行，方差减小（变得更加均匀）。
    因此我们使用负方差作为混合的衡量标准。"""
    return -np.var(concentration)


def compute_mixing_quality(concentration: np.ndarray) -> float:
    """计算杯子的混合程度。
    
    返回：
        值在 [0, 1] 中，其中 1 = 完美混合"""
    # 完美混合=所有像素具有相同的浓度
    mean_concentration = concentration.mean()
    variance = np.var(concentration)
    
    # 最大方差（对于该均值）是当一半为 0、一半为 2*均值时
    max_variance = mean_concentration * (1 - mean_concentration)
    
    if max_variance == 0:
        return 1.0
    
    return 1 - (variance / max_variance)


# 计算随时间变化的熵
print("Computing entropy over time...\n")

shannon_entropies = [compute_shannon_entropy(cup) for cup in mixing_history]
spatial_entropies = [compute_spatial_entropy(cup) for cup in mixing_history]
mixing_qualities = [compute_mixing_quality(cup) for cup in mixing_history]

# 绘制熵增长图
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 图 1：香农熵
axes[0].plot(shannon_entropies, linewidth=2, color='darkblue')
axes[0].set_xlabel('Time Step', fontsize=11)
axes[0].set_ylabel('Shannon Entropy (bits)', fontsize=11)
axes[0].set_title('Information Entropy Growth', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=shannon_entropies[0], color='red', linestyle='--', alpha=0.5, label='Initial')
axes[0].legend()

# 图 2：空间熵（负方差）
axes[1].plot(spatial_entropies, linewidth=2, color='darkgreen')
axes[1].set_xlabel('Time Step', fontsize=11)
axes[1].set_ylabel('Spatial Entropy (-Variance)', fontsize=11)
axes[1].set_title('Spatial Disorder Growth', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# 图 3：混合质量
axes[2].plot(mixing_qualities, linewidth=2, color='darkorange')
axes[2].set_xlabel('Time Step', fontsize=11)
axes[2].set_ylabel('Mixing Quality', fontsize=11)
axes[2].set_title('Approach to Equilibrium', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Perfect mixing')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Initial Shannon entropy: {shannon_entropies[0]:.3f} bits")
print(f"Final Shannon entropy:   {shannon_entropies[-1]:.3f} bits")
print(f"Entropy increase:        {shannon_entropies[-1] - shannon_entropies[0]:.3f} bits")
print(f"\nFinal mixing quality:    {mixing_qualities[-1]:.1%}")

print("\n🔑 Key insight: Entropy monotonically increases!")
print("   This is the second law of thermodynamics in action.")
print("\n✓ Entropy analysis complete!")

# 第三节：相空间和刘维尔定理

这里是深入的地方：**如果微观定律是可逆的，为什么熵会增加？**

## 相空间

考虑具有 $N$ 粒子的系统。 **相空间**是所有位置和动量的 6N 维空间：

$$\Gamma = (x_1, y_1, z_1, p_{x1}, p_{y1}, p_{z1}, \ldots, x_N, y_N, z_N, p_{xN}, p_{yN}, p_{zN})$$

相空间中的一个点对应一个**微观状态**，也就是对系统的完整描述。

## 刘维尔定理

**刘维尔定理**说相空间体积守恒：

$$\frac{d}{dt} \int_{V} d\Gamma = 0$$

这意味着：
- **微观上，动力学是可逆的**
- 相空间体积不变
- 信息得到保存

## 解决方案：粗粒化

关键是**粗粒化**：我们无法跟踪单个分子，因此我们将附近的微观状态分组为**宏观状态**。

- **微观状态**：所有分子的精确位置/动量（可逆）
- **宏观状态**：粗略描述，如“温度”或“浓度”（不可逆）

**熵增加**因为：
1. 许多微观状态映射到相同的宏观状态
2. 系统演化倾向于拥有更多微观状态的宏观状态
3. 进行粗粒化描述时，我们会丢失细节信息

我们来模拟一下吧！

In [ ]:
@dataclass
class Particle:
    '二维相空间中的粒子。'
    x: float
    y: float
    vx: float
    vy: float


def initialize_particles(num_particles: int = 100, region: str = 'left') -> List[Particle]:
    """初始化特定区域中的粒子。
    
    参数：
        num_particles：颗粒数
        region：盒子的“左”或“右”一半"""
    particles = []
    
    for _ in range(num_particles):
        if region == 'left':
            x = np.random.uniform(0.1, 0.4)
        else:
            x = np.random.uniform(0.6, 0.9)
        
        y = np.random.uniform(0.1, 0.9)
        
        # 随机速度
        speed = 0.02
        angle = np.random.uniform(0, 2*np.pi)
        vx = speed * np.cos(angle)
        vy = speed * np.sin(angle)
        
        particles.append(Particle(x, y, vx, vy))
    
    return particles


def update_particles(particles: List[Particle], dt: float = 1.0) -> List[Particle]:
    '更新粒子位置（具有反射边界的自由运动）。'
    new_particles = []
    
    for p in particles:
        # 更新位置
        new_x = p.x + p.vx * dt
        new_y = p.y + p.vy * dt
        new_vx, new_vy = p.vx, p.vy
        
        # 反映边界
        if new_x < 0 or new_x > 1:
            new_vx = -new_vx
            new_x = np.clip(new_x, 0, 1)
        
        if new_y < 0 or new_y > 1:
            new_vy = -new_vy
            new_y = np.clip(new_y, 0, 1)
        
        new_particles.append(Particle(new_x, new_y, new_vx, new_vy))
    
    return new_particles


def compute_macrostate(particles: List[Particle], num_bins: int = 4) -> np.ndarray:
    """将粗粒颗粒放入空间仓中。
    
    返回：
        颗粒计数的二维直方图"""
    positions = np.array([[p.x, p.y] for p in particles])
    
    hist, _, _ = np.histogram2d(
        positions[:, 0], positions[:, 1],
        bins=num_bins,
        range=[[0, 1], [0, 1]]
    )
    
    return hist


def compute_macrostate_entropy(macrostate: np.ndarray) -> float:
    '计算宏观状态的熵。'
    # 扁平化和标准化
    counts = macrostate.flatten()
    if counts.sum() == 0:
        return 0.0
    
    probs = counts / counts.sum()
    probs = probs[probs > 0]  # 删除零
    
    return -np.sum(probs * np.log2(probs))


# 模拟粒子混合
print("Simulating particle mixing with coarse-graining...\n")

# 初始化左侧的粒子
particles = initialize_particles(num_particles=200, region='left')

# 模拟
num_steps = 500
particle_history = [particles]
macrostate_history = []
macrostate_entropies = []

for step in range(num_steps):
    particles = update_particles(particles)
    particle_history.append([Particle(p.x, p.y, p.vx, p.vy) for p in particles])
    
    # 每 10 步计算一次宏观状态
    if step % 10 == 0:
        macrostate = compute_macrostate(particles)
        macrostate_history.append(macrostate)
        macrostate_entropies.append(compute_macrostate_entropy(macrostate))

# 可视化
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 顶行：微观状态（粒子位置）
timesteps_vis = [0, 100, 500]
for idx, t in enumerate(timesteps_vis):
    ax = axes[0, idx]
    ps = particle_history[t]
    positions = np.array([[p.x, p.y] for p in ps])
    
    ax.scatter(positions[:, 0], positions[:, 1], s=10, alpha=0.6, color='blue')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.set_title(f'Microstate at t={t}', fontweight='bold')
    ax.set_xlabel('x')
    ax.set_ylabel('y')

# 底行：宏观状态（粗粒化）
macro_timesteps = [0, 10, 50]
for idx, mt in enumerate(macro_timesteps):
    ax = axes[1, idx]
    im = ax.imshow(macrostate_history[mt].T, cmap='Blues', origin='lower', 
                   extent=[0, 1, 0, 1], vmin=0, vmax=macrostate_history[0].max())
    ax.set_title(f'Macrostate at t={mt*10}\nS={macrostate_entropies[mt]:.2f} bits', 
                fontweight='bold')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    plt.colorbar(im, ax=ax, label='Particle count')

plt.suptitle('Microscopic Reversibility vs Macroscopic Irreversibility', 
            fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# 绘制熵增长图
plt.figure(figsize=(10, 5))
plt.plot(np.arange(len(macrostate_entropies)) * 10, macrostate_entropies, 
         linewidth=2, color='darkred')
plt.xlabel('Time Step', fontsize=12)
plt.ylabel('Macroscopic Entropy (bits)', fontsize=12)
plt.title('Entropy Growth Despite Microscopic Reversibility', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("🔑 Key insight: Liouville's theorem guarantees microscopic reversibility.")
print("   But macroscopic entropy STILL increases due to coarse-graining!")
print("\n   • Microstate: Exact particle positions (reversible)")
print("   • Macrostate: Binned particle counts (irreversible)")
print("   • Many microstates → same macrostate")
print("   • Evolution explores more microstates over time")
print("\n✓ Phase space analysis complete!")

# 第 4 节：庞加莱回归——咖啡终会重新分离？

一个令人惊讶的事实是：**庞加莱回归定理**表明，有限系统最终会回到任意接近其初始状态的位置。

## 定理

对于**有限**相空间体积，几乎每个初始条件都会无限频繁地重复出现：

$$\lim_{t \to \infty} \inf \, d(\Gamma(t), \Gamma(0)) = 0$$

这意味着：
- 只要等待足够久，咖啡与牛奶就会**重新分离**！
- 粒子**将**返回到它们的起始配置！

## 问题在于：回归时间

你需要等待多长时间？对于 $N$ 个粒子：

$$t_{\text{recurrence}} \sim e^{N}$$

对于含有 $N \sim 10^{23}$ 个分子的咖啡：

$$t_{\text{recurrence}} \sim 10^{10^{23}} \text{ seconds}$$

这比宇宙的年龄（$\sim 10^{17}$ 秒）**长**长。

**实际不可逆性**：系统理论上是可逆的，但实际上是不可逆的。

让我们用一个小系统来演示这一点！

In [ ]:
def simple_phase_space_system(num_states: int = 8, num_steps: int = 10000) -> Tuple[List[int], List[int]]:
    """在有限相空间上模拟简单的确定性系统。
    
    参数：
        num_states：相空间的大小（小用于演示）
        num_steps：模拟的步骤数
    
    返回：
        states：访问过的州列表
        recurrence_times：返回初始状态的时间"""
    # 定义确定性演化规则
    # 简单示例： state_next = (a * state + b) mod num_states
    a, b = 3, 1  # 选择参数以提供有趣的动态
    
    initial_state = 0
    state = initial_state
    states = [state]
    recurrence_times = []
    
    for step in range(1, num_steps):
        # 确定性进化
        state = (a * state + b) % num_states
        states.append(state)
        
        # 检查是否复发
        if state == initial_state:
            recurrence_times.append(step)
    
    return states, recurrence_times


def estimate_recurrence_time_scaling() -> Tuple[List[int], List[float]]:
    '估计回归时间如何随系统大小变化。'
    sizes = [4, 8, 16, 32, 64, 128]
    avg_recurrence_times = []
    
    for size in sizes:
        _, rec_times = simple_phase_space_system(num_states=size, num_steps=size*20)
        if len(rec_times) > 0:
            avg_time = np.mean(np.diff([0] + rec_times))
        else:
            avg_time = size * 10  # 估计
        avg_recurrence_times.append(avg_time)
    
    return sizes, avg_recurrence_times


# 证明庞加莱回归
print("Demonstrating Poincaré recurrence...\n")

num_states = 16
states, recurrence_times = simple_phase_space_system(num_states=num_states, num_steps=200)

print(f"System with {num_states} states")
print(f"Initial state: {states[0]}")
print(f"\nRecurrences found at timesteps: {recurrence_times[:5]}...")

if len(recurrence_times) > 1:
    period = recurrence_times[1] - recurrence_times[0]
    print(f"Recurrence period: {period} steps")

# 可视化状态演化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 图 1：状态轨迹
ax = axes[0, 0]
ax.plot(states[:200], linewidth=1.5, color='darkblue')
for rt in recurrence_times:
    if rt < 200:
        ax.axvline(rt, color='red', alpha=0.3, linestyle='--')
ax.set_xlabel('Time Step', fontsize=11)
ax.set_ylabel('State', fontsize=11)
ax.set_title(f'State Evolution (Red lines = recurrence)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

# 图 2：状态直方图
ax = axes[0, 1]
ax.hist(states, bins=num_states, edgecolor='black', alpha=0.7, color='steelblue')
ax.set_xlabel('State', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('State Distribution (Should be uniform)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# 图 3：回归时间缩放
ax = axes[1, 0]
sizes, rec_times_avg = estimate_recurrence_time_scaling()
ax.semilogy(sizes, rec_times_avg, 'o-', linewidth=2, markersize=8, color='darkgreen', label='Observed')
# 拟合指数
log_rec_times = np.log(rec_times_avg)
coeffs = np.polyfit(sizes, log_rec_times, 1)
fit_line = np.exp(coeffs[1]) * np.exp(coeffs[0] * np.array(sizes))
ax.semilogy(sizes, fit_line, '--', linewidth=2, color='red', label=f'Exponential fit')
ax.set_xlabel('System Size (# states)', fontsize=11)
ax.set_ylabel('Average Recurrence Time', fontsize=11)
ax.set_title('Exponential Growth of Recurrence Time', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 图 4：外推到现实系统
ax = axes[1, 1]
N_values = np.array([10, 100, 1000, 10000, 1e23])  # 直到咖啡杯
t_rec_estimates = np.exp(N_values * 0.5)  # 非常粗略的估计
universe_age = 4.3e17  # 秒

ax.loglog(N_values[:-1], t_rec_estimates[:-1], 'o-', linewidth=2, markersize=8, color='purple')
ax.axhline(universe_age, color='red', linestyle='--', linewidth=2, label='Age of universe')
ax.set_xlabel('Number of Particles', fontsize=11)
ax.set_ylabel('Recurrence Time (seconds)', fontsize=11)
ax.set_title('Why Coffee Never Unmixes', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, which='both')
ax.text(100, 1e20, f'Coffee cup:\n$N \\sim 10^{{23}}$\n$t_{{rec}} \\sim 10^{{10^{{23}}}}$ s', 
        fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n🔑 Key insights:")
print("   1. Poincaré recurrence IS real - finite systems eventually recur")
print("   2. But recurrence time grows EXPONENTIALLY with system size")
print(f"   3. For coffee (~10²³ particles): t_rec ~ 10^(10^23) seconds")
print(f"   4. Universe age: ~10^17 seconds")
print("   5. So coffee will unmix... after waiting 10^(10^23) times the age of the universe!")
print("\n   This is why irreversibility is PRACTICAL even though dynamics are reversible.")
print("\n✓ Poincaré recurrence analysis complete!")

# 第五节：麦克斯韦妖——智能可以逆转熵吗？

**麦克斯韦妖**是一个挑战第二定律的思想实验。

## 思想实验的设置

1. 盒子被带门的隔板分成两个室
2. 气体分子随机运动
3. 一个“妖”负责控制小门：
   - 快速分子向右运动时开门
   - 慢速分子向左运动时开门
4. 结果：右边是热气体，左边是冷气体（熵减少！）

## 悖论

该妖似乎通过在不做功的情况下减少熵来**违反热力学第二定律**。

## 解决方案：兰道尔原理

这个妖必须**测量**分子速度并**存储**相关信息。为了持续运行，它最终必须**擦除**记忆。

**兰道尔原理** (1961)：擦除一位信息至少需要消散：

$$E_{\text{min}} = k_B T \ln 2$$

这些能量最终以热的形式耗散。

这种散热**增加了熵**，其增加量正是妖减少的量！

**深刻的含义**：**信息是物理的**。计算具有热力学成本。

让我们来模拟麦克斯韦妖吧！

In [ ]:
@dataclass
class GasParticle:
    '具有位置和速度的气体粒子。'
    x: float
    y: float
    vx: float
    vy: float
    
    def speed(self) -> float:
        return np.sqrt(self.vx**2 + self.vy**2)


class MaxwellsDemon:
    '模拟麦克斯韦的妖思想实验。'
    
    def __init__(self, num_particles: int = 100):
        self.num_particles = num_particles
        self.particles = self._initialize_particles()
        self.door_position = 0.5  # 盒子中间
        self.memory = []  # 妖的记忆
        self.entropy_cost = 0.0  # 擦除成本
    
    def _initialize_particles(self) -> List[GasParticle]:
        '使用麦克斯韦-玻尔兹曼分布初始化粒子。'
        particles = []
        for _ in range(self.num_particles):
            x = np.random.uniform(0.1, 0.9)
            y = np.random.uniform(0.1, 0.9)
            
            # 麦克斯韦-玻尔兹曼速度分布
            speed = np.random.rayleigh(0.02)
            angle = np.random.uniform(0, 2*np.pi)
            vx = speed * np.cos(angle)
            vy = speed * np.sin(angle)
            
            particles.append(GasParticle(x, y, vx, vy))
        return particles
    
    def update_without_demon(self, dt: float = 1.0):
        '更新粒子无魔（自然进化）。'
        new_particles = []
        for p in self.particles:
            new_x = p.x + p.vx * dt
            new_y = p.y + p.vy * dt
            new_vx, new_vy = p.vx, p.vy
            
            # 反映边界
            if new_x < 0 or new_x > 1:
                new_vx = -new_vx
                new_x = np.clip(new_x, 0, 1)
            if new_y < 0 or new_y > 1:
                new_vy = -new_vy
                new_y = np.clip(new_y, 0, 1)
            
            new_particles.append(GasParticle(new_x, new_y, new_vx, new_vy))
        
        self.particles = new_particles
    
    def update_with_demon(self, dt: float = 1.0, threshold_speed: float = 0.025):
        '用妖操作门来更新粒子。'
        new_particles = []
        
        for p in self.particles:
            new_x = p.x + p.vx * dt
            new_y = p.y + p.vy * dt
            new_vx, new_vy = p.vx, p.vy
            
            # 检查粒子是否穿过中间隔板
            crosses_door = (p.x < self.door_position <= new_x) or (p.x > self.door_position >= new_x)
            
            if crosses_door and 0.4 < new_y < 0.6:  # 门垂直位于中间
                # 妖测量速度并做出决定
                speed = p.speed()
                self.memory.append(speed)  # 存储测量（消耗内存）
                
                # 妖法则：
                # - 快速粒子向右移动
                # - 缓慢的粒子向左移动
                going_right = new_x > p.x
                is_fast = speed > threshold_speed
                
                if (going_right and not is_fast) or (not going_right and is_fast):
                    # 反射粒子（关门）
                    new_vx = -new_vx
                    new_x = p.x
            
            # 规则边界
            if new_x < 0 or new_x > 1:
                new_vx = -new_vx
                new_x = np.clip(new_x, 0, 1)
            if new_y < 0 or new_y > 1:
                new_vy = -new_vy
                new_y = np.clip(new_y, 0, 1)
            
            new_particles.append(GasParticle(new_x, new_y, new_vx, new_vy))
        
        self.particles = new_particles
        
        # 兰道尔擦除成本
        if len(self.memory) > 50:  # 抹去旧的记忆
            bits_erased = len(self.memory) - 50
            self.entropy_cost += bits_erased * np.log(2)  # k_B T ln(2) 每比特
            self.memory = self.memory[-50:]
    
    def compute_temperature_difference(self) -> float:
        '计算左室和右室之间的温差。'
        left_particles = [p for p in self.particles if p.x < self.door_position]
        right_particles = [p for p in self.particles if p.x >= self.door_position]
        
        if len(left_particles) == 0 or len(right_particles) == 0:
            return 0.0
        
        # 温度∝平均动能∝平均速度²
        left_temp = np.mean([p.speed()**2 for p in left_particles])
        right_temp = np.mean([p.speed()**2 for p in right_particles])
        
        return right_temp - left_temp


# 有妖和无妖的模拟
print("Simulating Maxwell's demon...\n")

# 没有妖
system_no_demon = MaxwellsDemon(num_particles=150)
temp_diffs_no_demon = []

for _ in range(300):
    system_no_demon.update_without_demon()
    temp_diffs_no_demon.append(system_no_demon.compute_temperature_difference())

# 与妖
system_with_demon = MaxwellsDemon(num_particles=150)
temp_diffs_with_demon = []
entropy_costs = []

for _ in range(300):
    system_with_demon.update_with_demon()
    temp_diffs_with_demon.append(system_with_demon.compute_temperature_difference())
    entropy_costs.append(system_with_demon.entropy_cost)

# 可视化
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 图1：无妖温差
axes[0, 0].plot(temp_diffs_no_demon, linewidth=2, color='blue', alpha=0.7)
axes[0, 0].axhline(0, color='black', linestyle='--', alpha=0.3)
axes[0, 0].set_xlabel('Time Step', fontsize=11)
axes[0, 0].set_ylabel('Temp Diff (Right - Left)', fontsize=11)
axes[0, 0].set_title('Without Demon: No Temperature Gradient', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# 图2：与妖的温差
axes[0, 1].plot(temp_diffs_with_demon, linewidth=2, color='red', alpha=0.7)
axes[0, 1].axhline(0, color='black', linestyle='--', alpha=0.3)
axes[0, 1].set_xlabel('Time Step', fontsize=11)
axes[0, 1].set_ylabel('Temp Diff (Right - Left)', fontsize=11)
axes[0, 1].set_title('With Demon: Temperature Gradient Created!', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# 图 3：粒子分布（没有妖的最终状态）
ax = axes[1, 0]
positions_no_demon = np.array([[p.x, p.y] for p in system_no_demon.particles])
speeds_no_demon = np.array([p.speed() for p in system_no_demon.particles])
scatter = ax.scatter(positions_no_demon[:, 0], positions_no_demon[:, 1], 
                    c=speeds_no_demon, s=20, cmap='coolwarm', alpha=0.6)
ax.axvline(0.5, color='black', linewidth=2, linestyle='--', label='Partition')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel('x', fontsize=11)
ax.set_ylabel('y', fontsize=11)
ax.set_title('Final State Without Demon', fontsize=12, fontweight='bold')
ax.legend()
plt.colorbar(scatter, ax=ax, label='Speed')

# 图 4：粒子分布（妖的最终状态）
ax = axes[1, 1]
positions_with_demon = np.array([[p.x, p.y] for p in system_with_demon.particles])
speeds_with_demon = np.array([p.speed() for p in system_with_demon.particles])
scatter = ax.scatter(positions_with_demon[:, 0], positions_with_demon[:, 1],
                    c=speeds_with_demon, s=20, cmap='coolwarm', alpha=0.6)
ax.axvline(0.5, color='black', linewidth=2, linestyle='--', label='Partition')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel('x', fontsize=11)
ax.set_ylabel('y', fontsize=11)
ax.set_title('Final State With Demon: Fast→Right, Slow→Left', fontsize=12, fontweight='bold')
ax.legend()
plt.colorbar(scatter, ax=ax, label='Speed')

plt.tight_layout()
plt.show()

# 兰道尔原理
plt.figure(figsize=(10, 5))
plt.plot(entropy_costs, linewidth=2, color='darkgreen')
plt.xlabel('Time Step', fontsize=12)
plt.ylabel('Cumulative Entropy Cost (k_B ln 2 per bit)', fontsize=12)
plt.title("Landauer's Principle: Information Erasure Has Entropic Cost", 
         fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n🔑 Key insights:")
print("   1. Demon CAN create temperature gradient (decrease entropy locally)")
print("   2. But demon must MEASURE and REMEMBER particle speeds")
print("   3. Memory is finite → must ERASE old measurements")
print(f"   4. Landauer: Erasing 1 bit releases ≥ k_B T ln(2) heat")
print(f"   5. Total entropy cost from erasure: {entropy_costs[-1]:.2f} k_B ln(2)")
print("   6. This EXACTLY compensates for entropy decrease!")
print("\n   → The second law is saved! Information is physical.")
print("\n✓ Maxwell's demon simulation complete!")

# 第 6 节：计算不可逆性 - 单向函数

不可逆性不仅存在于物理学中，也是**计算**的核心问题。

## 单向函数

如果满足以下条件，则函数 $f$ 是**单向**：
1. 易于计算：$y = f(x)$ 速度快
2. 难以反转：给定 $y$，找到 $x$ 使得 $f(x) = y$ 很难

**示例**：
- **乘法**：$f(p, q) = p \times q$（简单）
- **因式分解**：给定 $n = p \times q$，找到 $p, q$（对于大的 $n$ 很难）
- **密码学哈希**：例如 SHA-256

## 与热力学的联系

单向函数**计算上不可逆**：
- 计算 $f(x) \to y$ 时，部分信息会被**擦除或丢失**
- 许多输入 $x$ 映射到相同的输出 $y$
- 这就像热力学中的**粗粒化**！

## 兰道尔计算原理

**不可逆计算**（涉及信息擦除）存在最低能耗：

$$E_{\text{min}} = k_B T \ln(2) \times (\text{bits destroyed})$$

**可逆计算**（不擦除信息）原则上可以把能耗降到任意低。

这就是为什么量子计算机被设计为可逆的。

In [ ]:
def hash_function(x: int, num_bits_out: int = 8) -> int:
    """简单哈希函数（单向）。
    
    参数：
        x：输入整数
        num_bits_out：输出中的位数
    
    返回：
        [0, 2^num_bits_out - 1] 中的哈希值"""
    # 使用 Python 的内置哈希与模
    return hash(x) % (2 ** num_bits_out)


def cryptographic_hash(data: str) -> str:
    '使用 SHA-256 计算密码学哈希。'
    return hashlib.sha256(data.encode()).hexdigest()


def demonstrate_collision(num_inputs: int = 1000, num_bits_out: int = 8) -> dict:
    """演示哈希冲突：多个输入映射到同一个输出。"""
    hash_map = {}
    
    for x in range(num_inputs):
        h = hash_function(x, num_bits_out)
        if h not in hash_map:
            hash_map[h] = []
        hash_map[h].append(x)
    
    return hash_map


def compute_information_loss(input_bits: int, output_bits: int) -> float:
    """计算信息损失（以位为单位）。
    
    当将 n 位输入哈希为 m 位输出时 (m < n)，
    我们丢失了 (n - m) 位信息。"""
    return max(0, input_bits - output_bits)


# 演示单向函数
print("Demonstrating computational irreversibility...\n")

# 示例 1：简单的哈希冲突
print("Example 1: Hash Collisions")
print("="*50)
num_inputs = 1000
num_bits_out = 6  # 只有 64 个可能的输出
hash_map = demonstrate_collision(num_inputs, num_bits_out)

print(f"Hashed {num_inputs} inputs to {2**num_bits_out} possible outputs")
print(f"Number of collisions: {sum(1 for v in hash_map.values() if len(v) > 1)}")
print(f"Average inputs per output: {num_inputs / len(hash_map):.1f}")

# 展示部分哈希冲突
collision_example = [v for v in hash_map.values() if len(v) > 2][0]
print(f"\nExample collision: Inputs {collision_example[:5]} all hash to {hash_function(collision_example[0], num_bits_out)}")

# 示例 2：密码学哈希
print("\n" + "="*50)
print("Example 2: Cryptographic Hash (SHA-256)")
print("="*50)

messages = [
    "Hello, World!",
    "Hello, World.",  # 一个字符不同
    "The coffee automaton demonstrates irreversibility"
]

for msg in messages:
    hash_val = cryptographic_hash(msg)
    print(f"Message: '{msg[:40]}'")
    print(f"SHA-256: {hash_val[:32]}...")
    print()

print("Note: Tiny change in input → completely different hash (avalanche effect)")
print("This makes inversion computationally infeasible!")

# 可视化哈希分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 图 1：哈希冲突分布
ax = axes[0]
collision_counts = [len(v) for v in hash_map.values()]
ax.hist(collision_counts, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
ax.set_xlabel('Number of Inputs per Hash Value', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Hash Collisions: Many-to-One Mapping', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.axvline(np.mean(collision_counts), color='red', linestyle='--', linewidth=2,
          label=f'Mean: {np.mean(collision_counts):.1f}')
ax.legend()

# 图 2：信息丢失
ax = axes[1]
input_bits_range = np.arange(8, 65, 4)
output_bits_fixed = 16
info_loss = [compute_information_loss(ib, output_bits_fixed) for ib in input_bits_range]

ax.plot(input_bits_range, info_loss, 'o-', linewidth=2, markersize=8, color='darkred')
ax.fill_between(input_bits_range, 0, info_loss, alpha=0.3, color='red')
ax.set_xlabel('Input Size (bits)', fontsize=11)
ax.set_ylabel('Information Lost (bits)', fontsize=11)
ax.set_title(f'Information Loss in Hashing (→{output_bits_fixed} bits)', 
            fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 兰道尔成本
print("\n" + "="*50)
print("Landauer's Principle for Computation")
print("="*50)

bits_destroyed = 48  # 将 64 位输入哈希为 16 位输出
k_B = 1.380649e-23  # 玻尔兹曼常数 (J/K)
T = 300  # 温度（K）
energy_per_bit = k_B * T * np.log(2)  # 焦耳
total_energy = bits_destroyed * energy_per_bit

print(f"Hashing 64-bit input → 16-bit output:")
print(f"  Bits destroyed: {bits_destroyed}")
print(f"  Minimum energy cost: {total_energy:.2e} J")
print(f"  At T=300K: {energy_per_bit:.2e} J per bit")
print(f"\nModern CPUs use ~10^-9 J per operation (well above Landauer limit)")
print(f"Landauer limit: {energy_per_bit:.2e} J per bit erased")
print(f"Gap: ~10^6× above minimum! Room for improvement.")

print("\n🔑 Key insights:")
print("   1. One-way functions destroy information (irreversible)")
print("   2. Many inputs map to same output (collisions)")
print("   3. This is computational coarse-graining!")
print("   4. Landauer: Destroying 1 bit costs ≥ k_B T ln(2) energy")
print("   5. Reversible computation (no info loss) could be free!")
print("\n✓ Computational irreversibility demonstration complete!")

# 第 7 节：机器学习和信息瓶颈

**不可逆性对于机器学习至关重要！**

## 为什么神经网络需要遗忘

神经网络是一个**有损压缩**函数：

$$f: \mathbb{R}^{d_{\text{in}}} \to \mathbb{R}^{d_{\text{out}}}$$

其中通常为 $d_{\text{out}} \ll d_{\text{in}}$。

**信息瓶颈**：隐藏层压缩输入，丢弃不相关的细节：
- 输入：高维数据，例如图像像素
- 隐藏层：渐进压缩
- 输出：低维表示，例如类别标签

## 为什么压缩有帮助

1. **泛化**：忽略噪声，保留有效信号
2. **效率**：仅存储相关特征
3. **鲁棒性**：相似的输入→相似的输出

**与热力学的联系**：
- **不可逆压缩** = 丢失信息
- **熵增加** = 传播不确定性
- **粗粒化** = 对相似的输入进行分组

## 信息瓶颈原理

寻找表示 $T$：
$$\min_{T} [I(X; T) - \beta \cdot I(T; Y)]$$

- $I(X; T)$：$T$ 记住有关输入 $X$ 的信息（压缩）
- $I(T; Y)$：$T$ 保留有关输出 $Y$ 的信息（预测）
- $\beta$：权衡参数

**目标**：最大限度地压缩，同时保持预测能力。

让我们来演示一下！

In [ ]:
def create_autoencoder_layers(input_dim: int, hidden_dims: List[int]) -> List[Tuple[np.ndarray, np.ndarray]]:
    """创建一个简单的自编码器（压缩然后解压缩）。
    
    返回：
        每层的 (W, b) 列表"""
    layers = []
    dims = [input_dim] + hidden_dims + [input_dim]
    
    for i in range(len(dims) - 1):
        W = np.random.randn(dims[i], dims[i+1]) * np.sqrt(2.0 / dims[i])
        b = np.zeros(dims[i+1])
        layers.append((W, b))
    
    return layers


def forward_autoencoder(x: np.ndarray, layers: List[Tuple]) -> Tuple[np.ndarray, List[np.ndarray]]:
    """通过自编码器前向传播。
    
    返回：
        重建，每层的激活列表"""
    activations = [x]
    current = x
    
    for i, (W, b) in enumerate(layers):
        current = current @ W + b
        # ReLU 用于隐藏层，线性输出
        if i < len(layers) - 1:
            current = np.maximum(0, current)
        activations.append(current)
    
    return current, activations


def measure_information_content(activations: np.ndarray, num_bins: int = 20) -> float:
    '通过激活分布熵估计信息内容。'
    # 扁平化激活
    flat = activations.flatten()
    
    # 创建直方图
    hist, _ = np.histogram(flat, bins=num_bins, density=True)
    hist = hist / hist.sum()
    hist = hist[hist > 0]
    
    # 香农熵
    return -np.sum(hist * np.log2(hist))


# 生成合成数据
print("Demonstrating information bottleneck in neural networks...\n")

# 创建具有低维结构的高维输入
num_samples = 500
input_dim = 50
latent_dim = 3  # 真实的底层维度

# 生成数据：低维潜变量 → 高维观测
latent = np.random.randn(num_samples, latent_dim)
projection = np.random.randn(latent_dim, input_dim)
X = latent @ projection
X += np.random.randn(num_samples, input_dim) * 0.5  # 添加噪音

print(f"Generated data:")
print(f"  Samples: {num_samples}")
print(f"  Input dimension: {input_dim}")
print(f"  True latent dimension: {latent_dim}")
print(f"  Information must be compressed {input_dim/latent_dim:.1f}×!\n")

# 创建具有瓶颈的自编码器
hidden_dims = [25, 10, 5]  # 逐步压缩
layers = create_autoencoder_layers(input_dim, hidden_dims)

print(f"Autoencoder architecture: {input_dim} → {' → '.join(map(str, hidden_dims))} → {input_dim}")
print(f"Bottleneck: {hidden_dims[-1]} dimensions (compression: {input_dim/hidden_dims[-1]:.1f}×)\n")

# 分析信息流
reconstructions = []
all_activations = []

for x in X[:100]:  # 使用子集来提高速度
    recon, acts = forward_autoencoder(x, layers)
    reconstructions.append(recon)
    all_activations.append(acts)

# 测量每一层的信息
layer_entropies = []
layer_sizes = [input_dim] + hidden_dims + [input_dim]

for layer_idx in range(len(layers) + 1):
    layer_activations = np.array([acts[layer_idx] for acts in all_activations])
    entropy = measure_information_content(layer_activations)
    layer_entropies.append(entropy)

# 可视化信息瓶颈
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 图1：网络架构
ax = axes[0, 0]
x_pos = np.arange(len(layer_sizes))
ax.bar(x_pos, layer_sizes, color=['blue' if s > hidden_dims[-1] else 'red' for s in layer_sizes],
       alpha=0.7, edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels(['Input'] + [f'H{i+1}' for i in range(len(hidden_dims))] + ['Output'])
ax.set_ylabel('Layer Dimension', fontsize=11)
ax.set_title('Network Architecture: Information Bottleneck', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# 图 2：每层的信息内容
ax = axes[0, 1]
ax.plot(x_pos, layer_entropies, 'o-', linewidth=2, markersize=10, color='darkgreen')
ax.fill_between(x_pos, 0, layer_entropies, alpha=0.3, color='green')
ax.set_xticks(x_pos)
ax.set_xticklabels(['Input'] + [f'H{i+1}' for i in range(len(hidden_dims))] + ['Output'])
ax.set_ylabel('Information Content (bits)', fontsize=11)
ax.set_title('Information Loss Through Network', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.axvline(len(hidden_dims), color='red', linestyle='--', linewidth=2, label='Bottleneck')
ax.legend()

# 图 3：压缩比
ax = axes[1, 0]
compression_ratios = [layer_sizes[0] / s for s in layer_sizes]
ax.bar(x_pos, compression_ratios, color='orange', alpha=0.7, edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels(['Input'] + [f'H{i+1}' for i in range(len(hidden_dims))] + ['Output'])
ax.set_ylabel('Compression Ratio', fontsize=11)
ax.set_title('Lossy Compression Through Network', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(1, color='black', linestyle='--', alpha=0.5)

# 图 4：维度与信息
ax = axes[1, 1]
ax.scatter(layer_sizes, layer_entropies, s=200, c=range(len(layer_sizes)), 
          cmap='viridis', edgecolor='black', linewidth=2)
for i, (d, e) in enumerate(zip(layer_sizes, layer_entropies)):
    label = 'Input' if i == 0 else f'H{i}' if i <= len(hidden_dims) else 'Output'
    ax.annotate(label, (d, e), xytext=(5, 5), textcoords='offset points', fontsize=10)
ax.set_xlabel('Layer Dimension', fontsize=11)
ax.set_ylabel('Information Content (bits)', fontsize=11)
ax.set_title('Dimension vs Information Content', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔑 Key insights:")
print(f"   1. Input dimension: {input_dim}, Information: {layer_entropies[0]:.2f} bits")
print(f"   2. Bottleneck dimension: {hidden_dims[-1]}, Information: {layer_entropies[len(hidden_dims)]:.2f} bits")
print(f"   3. Information lost: {layer_entropies[0] - layer_entropies[len(hidden_dims)]:.2f} bits ({(1 - layer_entropies[len(hidden_dims)]/layer_entropies[0])*100:.1f}%)")
print(f"   4. This is INTENTIONAL! Network forgets noise, remembers structure.")
print("\n   → Irreversibility (information loss) is essential for learning!")
print("   → Networks that compress well generalize well.")
print("   → Thermodynamic analogy: Compress = coarse-grain = increase entropy")
print("\n✓ Information bottleneck demonstration complete!")

# 第 8 节：时间之箭——基本规律还是涌现现象

我们从很多角度都看到了不可逆性。现在让我们问一个深刻的问题：

**时间之箭是基本规律，还是涌现现象？**

## 支持“基本规律”的论据

1. **弱相互作用违反 CP 对称性** → 暗示微弱的时间反演不对称
2. **宇宙学箭头**：宇宙膨胀（不收缩）
3. **量子测量**：波函数塌缩是不可逆的

## 支持“涌现现象”的论据

1. **统计力学**：第二定律源于统计
2. **微观可逆性**：基本定律是时间对称的
3. **边界条件**：低熵大爆炸设定时间箭头

## 主流观点：时间之箭主要是涌现的

时间之箭**不在**物理定律中——而是在**初始条件**中：

$$S_{\text{universe}}(t) > S_{\text{universe}}(0)$$

大爆炸始于**极低的熵**。从那时起，熵一直在增加。

**为什么大爆炸是低熵的？**我们不知道！这是物理学中最深奥的谜团之一。

## 影响

- **热力学箭头**：熵增加
- **心理箭头**：我们记住过去，而不是未来（记忆需要低熵）
- **宇宙学箭头**：宇宙膨胀

所有这三个都是低熵初始条件的**后果**。

让我们想象一下这一点！

In [ ]:
# 可视化时间箭头

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 图 1：宇宙的熵与时间
ax = axes[0, 0]
time_universe = np.linspace(0, 13.8, 1000)  # 十亿年
# 简化模型：熵随时间呈对数增长
entropy_universe = np.log(1 + time_universe * 10) + np.random.randn(1000) * 0.1

ax.plot(time_universe, entropy_universe, linewidth=2, color='darkblue')
ax.scatter([0], [entropy_universe[0]], s=200, c='red', marker='*', 
          label='Big Bang (low entropy!)', zorder=5, edgecolor='black', linewidth=2)
ax.arrow(3, 1, 3, 1, head_width=0.2, head_length=0.5, fc='red', ec='red', linewidth=2)
ax.text(4, 2.5, 'Arrow of Time', fontsize=12, color='red', fontweight='bold')
ax.set_xlabel('Time (billion years since Big Bang)', fontsize=11)
ax.set_ylabel('Universe Entropy (arbitrary units)', fontsize=11)
ax.set_title('Cosmological Arrow of Time', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 图 2：时间上的向前与向后
ax = axes[0, 1]
t = np.linspace(0, 10, 100)
entropy_forward = 1 - np.exp(-t/3)  # 熵增加
entropy_backward = np.flip(entropy_forward)  # 时间反转（熵减少）

ax.plot(t, entropy_forward, linewidth=3, color='green', label='Forward in time (2nd law holds)')
ax.plot(t, entropy_backward, linewidth=3, color='red', linestyle='--', 
       label='Backward in time (2nd law violated!)')
ax.set_xlabel('Time', fontsize=11)
ax.set_ylabel('Entropy', fontsize=11)
ax.set_title('Time Asymmetry: Why Backward Looks Wrong', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 图 3：相空间体积 (Liouville) 与粗粒熵
ax = axes[1, 0]
t = np.linspace(0, 10, 100)
phase_space_volume = np.ones_like(t)  # 常数（刘维尔）
macro_entropy = 1 - np.exp(-t/2)  # 增加

ax2 = ax.twinx()
ax.plot(t, phase_space_volume, linewidth=3, color='blue', label='Microscopic (Liouville)')
ax2.plot(t, macro_entropy, linewidth=3, color='red', label='Macroscopic (2nd law)')
ax.set_xlabel('Time', fontsize=11)
ax.set_ylabel('Phase Space Volume (conserved)', fontsize=11, color='blue')
ax2.set_ylabel('Macroscopic Entropy (increases)', fontsize=11, color='red')
ax.set_title('Microscopic Reversibility vs Macroscopic Irreversibility', 
            fontsize=12, fontweight='bold')
ax.tick_params(axis='y', labelcolor='blue')
ax2.tick_params(axis='y', labelcolor='red')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='center right')
ax.grid(True, alpha=0.3)

# 情节4：时间的三箭
ax = axes[1, 1]
ax.axis('off')

# 画三个箭头
arrow_props = dict(arrowstyle='->', lw=4, color='darkblue')
ax.annotate('', xy=(0.8, 0.8), xytext=(0.2, 0.8), arrowprops=arrow_props)
ax.text(0.5, 0.85, 'Thermodynamic Arrow', ha='center', fontsize=12, fontweight='bold')
ax.text(0.5, 0.75, 'Entropy increases', ha='center', fontsize=10, style='italic')

arrow_props['color'] = 'darkgreen'
ax.annotate('', xy=(0.8, 0.5), xytext=(0.2, 0.5), arrowprops=arrow_props)
ax.text(0.5, 0.55, 'Psychological Arrow', ha='center', fontsize=12, fontweight='bold')
ax.text(0.5, 0.45, 'Remember past, not future', ha='center', fontsize=10, style='italic')

arrow_props['color'] = 'darkred'
ax.annotate('', xy=(0.8, 0.2), xytext=(0.2, 0.2), arrowprops=arrow_props)
ax.text(0.5, 0.25, 'Cosmological Arrow', ha='center', fontsize=12, fontweight='bold')
ax.text(0.5, 0.15, 'Universe expands', ha='center', fontsize=10, style='italic')

ax.text(0.5, 0.05, 'All three aligned → All consequences of low-entropy Big Bang', 
       ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title('The Three Arrows of Time', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n🔑 Deep insights about time:")
print("\n1. MICROSCOPIC LAWS: Time-reversible (Newtonian, quantum mechanics)")
print("   → Running laws backward is mathematically valid")
print("\n2. MACROSCOPIC BEHAVIOR: Time-irreversible (thermodynamics)")
print("   → Entropy increases, mixing happens, cannot unmix")
print("\n3. THE RESOLUTION: Initial conditions + statistics")
print("   → Big Bang had EXTREMELY low entropy")
print("   → Statistical evolution explores high-entropy states")
print("   → Coarse-graining makes evolution appear irreversible")
print("\n4. THE DEEP MYSTERY: Why was the Big Bang low-entropy?")
print("   → We don't know! This is the origin of the arrow of time.")
print("   → Some theories: anthropic principle, bouncing cosmology, etc.")
print("\n5. THREE ARROWS, ONE CAUSE:")
print("   → Thermodynamic: Entropy ↑ (consequence of initial conditions)")
print("   → Psychological: Memory works backward (requires low entropy)")
print("   → Cosmological: Universe expands (related to initial conditions)")
print("\n✓ Arrow of time analysis complete!")

# 第 9 节：生物不可逆性——生命与热力学第二定律

**生命是否违反第二定律？**

不会。生命是一个**开放系统**：
1. 减少局部熵（创建秩序）
2. 增加环境及总体熵（向外输出无序）

## Schrödinger 的洞见

在《生命是什么？》 （1944），薛定谔说：
> “生命以负熵为食”

生物体：
- 导入**低熵**能量（食物、阳光）
- 用它来维持秩序（新陈代谢、生长、繁殖）
- 向外排出**高熵**废物，例如热量和二氧化碳

**最终结果**：宇宙熵增加，遵守第二定律。

## 开放系统第二定律

对于与环境交换熵 $S_{\text{env}}$ 的开放系统：

$$\frac{dS_{\text{system}}}{dt} + \frac{dS_{\text{env}}}{dt} \geq 0$$

即使 $S_{\text{system}}$ 减少，只要 $S_{\text{env}}$ 增加得更多，总熵仍然会增加。

我们来建模一下吧！

In [ ]:
# 建立一个维持秩序的简单“生命”系统模型

def simulate_open_system(num_steps: int = 200) -> dict:
    """模拟一个保持低熵的开放系统。
    
    返回：
        具有熵历史的字典"""
    # 初始化
    S_system = 10.0  # 系统以中等熵开始
    S_environment = 0.0  # 跟踪导出的累积熵
    
    S_system_history = [S_system]
    S_environment_history = [S_environment]
    S_total_history = [S_system + S_environment]
    
    for t in range(num_steps):
        # 系统进程：自然会增加熵
        natural_increase = 0.5
        
        # 主动维护：系统向环境输出熵
        # 这在环境中“消耗”的熵比在系统中节省的熵更多
        entropy_exported = 0.8  # 超过自然增长（第二定律！）
        entropy_reduced = 0.3  # 系统熵减少
        
        # 更新
        S_system = S_system + natural_increase - entropy_reduced
        S_environment = S_environment + entropy_exported
        
        # 记录
        S_system_history.append(S_system)
        S_environment_history.append(S_environment)
        S_total_history.append(S_system + S_environment)
    
    return {
        'system': S_system_history,
        'environment': S_environment_history,
        'total': S_total_history
    }


def simulate_closed_system(num_steps: int = 200) -> dict:
    '模拟封闭系统（用于比较）。'
    S_total = 10.0
    S_total_history = [S_total]
    
    for t in range(num_steps):
        # 熵只在封闭系统中增加
        S_total = S_total + 0.5
        S_total_history.append(S_total)
    
    return {'total': S_total_history}


print("Simulating open system (life-like) vs closed system...\n")

open_sys = simulate_open_system()
closed_sys = simulate_closed_system()

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 图 1：开放系统（生命）
ax = axes[0]
t = np.arange(len(open_sys['total']))
ax.plot(t, open_sys['system'], linewidth=2, label='System (organism)', color='green')
ax.plot(t, open_sys['environment'], linewidth=2, label='Environment', color='brown')
ax.plot(t, open_sys['total'], linewidth=3, label='Total (system + env)', 
       color='red', linestyle='--')
ax.set_xlabel('Time', fontsize=11)
ax.set_ylabel('Entropy', fontsize=11)
ax.set_title('Open System: Life Maintains Order by Exporting Entropy', 
            fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.text(100, 50, 'System entropy\ncan decrease!', fontsize=10, color='green',
       bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
ax.text(100, 120, 'But total entropy\nalways increases!', fontsize=10, color='red',
       bbox=dict(boxstyle='round', facecolor='pink', alpha=0.5))

# 图 2：与封闭系统的比较
ax = axes[1]
ax.plot(t, closed_sys['total'], linewidth=3, label='Closed system', color='blue')
ax.plot(t, open_sys['total'], linewidth=3, label='Open system (total)', 
       color='red', linestyle='--')
ax.set_xlabel('Time', fontsize=11)
ax.set_ylabel('Total Entropy', fontsize=11)
ax.set_title('Both Systems Obey 2nd Law: Total Entropy ↑', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.text(100, 50, 'Open system increases\nentropy FASTER\n(due to metabolism)', 
       fontsize=10, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))

plt.tight_layout()
plt.show()

print("\n🔑 Key insights about life and entropy:")
print("\n1. LOCAL vs GLOBAL:")
print(f"   → System entropy: {open_sys['system'][0]:.1f} → {open_sys['system'][-1]:.1f} (maintained low!)")
print(f"   → Total entropy:  {open_sys['total'][0]:.1f} → {open_sys['total'][-1]:.1f} (increased!)")
print("\n2. LIFE'S TRICK:")
print("   → Import low-entropy energy (food, sunlight)")
print("   → Use it to build order (proteins, cells, organisms)")
print("   → Export high-entropy waste (heat, CO₂, etc.)")
print("\n3. NET RESULT:")
print("   → Organism entropy ↓ (more order)")
print("   → Environment entropy ↑↑ (much more disorder)")
print("   → Total entropy ↑ (2nd law satisfied!)")
print("\n4. WHY IT WORKS:")
print("   → Exporting entropy is IRREVERSIBLE")
print("   → Heat cannot spontaneously reconcentrate")
print("   → This is why death is inevitable (eventually can't export enough)")
print("\n5. SCHRÖDINGER WAS RIGHT:")
print("   → Life feeds on negative entropy (order)")
print("   → Exports positive entropy (disorder)")
print("   → This is how life is compatible with 2nd law!")
print("\n✓ Biological irreversibility demonstration complete!")

# 第 10 节：综合——不同尺度上的不可逆性

我们探索了从咖啡杯到宇宙的不可逆性。我们来综合一下。

## 不同尺度上的不可逆性

| 尺度 | 系统 | 是否可逆？ | 实际不可逆的原因 |
|-------|--------|-------------|-------------------|
| **微观** | 单个粒子 | ✅ 是 | 牛顿力学和量子定律具有时间对称性 |
| **介观** | 小系统（约 100 个粒子） | ⚠️ 几乎不可逆 | 庞加莱回归时间远大于观察时间 |
| **宏观** | 日常物体 | ❌ 否 | 粗粒化、统计效应和巨大的粒子数 N |
| **宇宙学** | 宇宙 | ❌ 否 | 低熵初始条件 |

## 共同特征

所有形式的不可逆性都具有以下特点：

1. **粗粒化**：丢失细粒度信息
   - 物理学：宏观状态与微观状态
   - 计算：哈希函数、有损压缩
   - ML：神经网络瓶颈

2. **多对一映射**：多个输入→相同的输出
   - 咖啡：许多分子配置→相同的宏观外观
   - 哈希：许多字符串→相同的哈希值
   - 神经网络：许多图像→同一类标签

3. **信息丢失**：无法恢复原始状态
   - 热力学：熵增
   - Landauer：位擦除会消耗能源
   - ML：压缩丢弃细节

4. **统计倾向**：系统向高概率状态演化
   - 物理学：最有可能达到平衡
   - 计算：状态空间中的随机游走
   - ML：向最小值梯度下降

## 深远的影响

### 对于物理
- 时间是涌现的，而不是根本的
- 第二定律是统计性的，不是绝对的
- 低熵初始条件是关键谜团

### 对计算的意义
- 所有计算都有热力学成本
- 可逆计算可能是“免费的”
- 信息是物理的（不是抽象的）

### 对于机器学习
- 压缩对于泛化至关重要
- 信息瓶颈可以形成良好的归纳偏置
- 遗忘（不可逆性）有助于学习

### 对生命的意义
- 开放系统可以维持秩序
- 但必须输出熵
- 死亡是热力学必然性

## 终极问题

**为什么宇宙诞生时的熵这么低？**

这个事实解释了：
- 时间之箭
- 为什么混合是不可逆的
- 为什么我们记住过去而不是未来
- 生命为何能够存在
- 为什么计算是可能的

我们不知道答案。但我们知道这是科学中最深刻的问题之一。

---

## 与机器学习的联系（重温）

**为什么这对人工智能很重要？**

1. **信息瓶颈**：神经网络必须压缩→不可逆
2. **单向函数**：安全性取决于计算的不可逆性
3. **兰道尔极限**：未来人工智能能源效率受热力学限制
4. **记忆**：大脑/计算机必须删除旧记忆→熵成本
5. **学习=压缩**：好的模型不可逆地压缩数据

**不可逆性不是缺陷，而是一种功能。**

没有它：
- 无法泛化（系统会记住一切）
- 没有计算（没有单向函数）
- 没有安全性（任何单向函数都可能被轻易逆转）
- 没有学习（没有抽象）

---

**咖啡自动机告诉我们**：宇宙具有时间箭头，它指向更高的熵、被遗忘的信息和不可逆的变化。

**但矛盾的是**，这种不可逆性使得：
- 计算成为可能
- 生命成为可能
- 学习成为可能
- 思考成为可能

你无法让混合后的咖啡自行重新分离；但正是这种不可逆性，使你能够思考这个问题。

In [ ]:
# 最终可视化：完整图片

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 中心概念：时间之箭
ax_center = fig.add_subplot(gs[1, 1])
ax_center.axis('off')
ax_center.text(0.5, 0.5, '☕\n\nThe Coffee Automaton\n\nIrreversibility', 
              ha='center', va='center', fontsize=20, fontweight='bold',
              bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.8))
ax_center.set_xlim(0, 1)
ax_center.set_ylim(0, 1)

# 周边概念
concepts = [
    ("Physics\n\nEntropy ↑\n2nd Law", gs[0, 0]),
    ("Phase Space\n\nLiouville\nCoarse-graining", gs[0, 1]),
    ("Computation\n\nOne-way\nLandauer", gs[0, 2]),
    ("Recurrence\n\nPoincaré\ne^N time", gs[1, 0]),
    ("Maxwell\n\nDemon\nInfo=Physical", gs[1, 2]),
    ("Biology\n\nLife\nOpen system", gs[2, 0]),
    ("ML\n\nCompression\nBottleneck", gs[2, 1]),
    ("Cosmology\n\nBig Bang\nLow entropy", gs[2, 2]),
]

colors = ['lightcoral', 'lightgreen', 'lightyellow', 'lightpink', 
         'lightcyan', 'lavender', 'peachpuff', 'thistle']

for (text, pos), color in zip(concepts, colors):
    ax = fig.add_subplot(pos)
    ax.axis('off')
    ax.text(0.5, 0.5, text, ha='center', va='center', fontsize=12,
           bbox=dict(boxstyle='round', facecolor=color, alpha=0.7))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    
    # 将箭头绘制到中心
    ax.annotate('', xy=(0.5, 0.5), xytext=(0.5, 0.5),
               arrowprops=dict(arrowstyle='->', lw=2, color='gray', alpha=0.5))

plt.suptitle('Irreversibility: A Unifying Concept Across All Scales', 
            fontsize=16, fontweight='bold', y=0.98)
plt.show()

# 统计汇总
print("\n" + "="*70)
print("SUMMARY: THE COFFEE AUTOMATON - KEY TAKEAWAYS")
print("="*70)

print("\n1. FUNDAMENTAL PUZZLE:")
print("   • Microscopic laws are reversible (Newton, Schrödinger)")
print("   • Macroscopic behavior is irreversible (coffee mixes, never unmixes)")
print("   • Resolution: Coarse-graining + statistics + low-entropy initial condition")

print("\n2. MECHANISMS OF IRREVERSIBILITY:")
print("   • Coarse-graining: Losing information when grouping microstates")
print("   • Statistical mechanics: High-entropy states vastly outnumber low-entropy")
print("   • Poincaré recurrence: Reversible, but on timescale e^N >> universe age")

print("\n3. INFORMATION IS PHYSICAL:")
print("   • Landauer's principle: Erasing 1 bit costs k_B T ln(2) energy")
print("   • Maxwell's demon: Information gathering/erasure has entropic cost")
print("   • Computation: All irreversible operations dissipate heat")

print("\n4. COMPUTATIONAL IRREVERSIBILITY:")
print("   • One-way functions: Easy forward, hard backward")
print("   • Cryptographic hashing: Many inputs → same output")
print("   • Security depends on computational irreversibility")

print("\n5. MACHINE LEARNING:")
print("   • Neural networks compress (information bottleneck)")
print("   • Compression = irreversible = forgetting details")
print("   • Generalization requires irreversibility!")

print("\n6. LIFE AND THERMODYNAMICS:")
print("   • Life is open system: imports order, exports disorder")
print("   • Local entropy ↓, but total entropy ↑ (2nd law satisfied)")
print("   • Death is thermodynamic inevitability")

print("\n7. THE ARROW OF TIME:")
print("   • Not in the laws—in the initial conditions!")
print("   • Big Bang had extremely low entropy (why??)")
print("   • All three arrows (thermodynamic, psychological, cosmological) aligned")

print("\n8. DEEP MYSTERY:")
print("   • Why was the Big Bang low-entropy?")
print("   • This single fact explains the arrow of time")
print("   • Still unsolved! One of deepest questions in physics")

print("\n" + "="*70)
print("\n☕ The coffee automaton: Simple system, profound implications")
print("\nYou can't unmix the coffee.")
print("But this irreversibility is what makes computation, life, and thought possible.")
print("\n" + "="*70)
print("\n✓ Complete analysis of irreversibility finished!")
print("\n🎓 Paper 19 implementation complete: A deep dive into the coffee automaton.")